# Alfido Tech | Task 3: Website Traffic Analysis
**Source dataset:** Kaggle Website Traffic Analysis (uploaded file: `traffic.csv`)  
**Analysis period in source:** 19–25 August 2021

## Important source-data limitation
The supplied file contains event-level music-link activity (`pageview`, `click`, `preview`) and fields for date, country, city, artist, album, track, ISRC and link ID. It does **not** contain user IDs, session IDs, event timestamps, page URLs, referrer/source fields or duration fields. Therefore sessions, unique users, bounce rate, average session duration, landing/exit pages and true user journeys cannot be calculated from this source. This notebook reports valid event-level engagement measures and clearly flags unavailable metrics rather than inventing them.

In [ ]:
# 1. Load the dataset (CSV or ZIP)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import zipfile, io

CSV_NAME = "traffic.csv"
ZIP_NAME = "archive.zip"

def load_data():
    if Path(CSV_NAME).exists():
        return pd.read_csv(CSV_NAME)
    if Path(ZIP_NAME).exists():
        with zipfile.ZipFile(ZIP_NAME) as z:
            csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
            if not csvs:
                raise FileNotFoundError("No CSV file found in archive.zip")
            return pd.read_csv(io.BytesIO(z.read(csvs[0])))
    try:
        from google.colab import files
        print("Upload traffic.csv or the downloaded ZIP file.")
        uploaded = files.upload()
        name = next(iter(uploaded))
        if name.lower().endswith(".zip"):
            with zipfile.ZipFile(io.BytesIO(uploaded[name])) as z:
                csvs = [n for n in z.namelist() if n.lower().endswith(".csv")]
                return pd.read_csv(io.BytesIO(z.read(csvs[0])))
        return pd.read_csv(io.BytesIO(uploaded[name]))
    except ImportError:
        raise FileNotFoundError("Place traffic.csv or archive.zip beside this notebook.")

df = load_data()
print("Raw shape:", df.shape)
display(df.head())

In [ ]:
# 2. Parse and clean event/log data
required = ["event", "date", "linkid"]
missing_required = [c for c in required if c not in df.columns]
if missing_required:
    raise ValueError(f"Required columns missing: {missing_required}")

print("Missing values before cleaning:\n", df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())

df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["event", "date", "linkid"]).copy()
for col in ["country", "city", "artist", "album", "track", "isrc"]:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown").astype(str)
df["event"] = df["event"].astype(str).str.strip().str.lower()

print("Cleaned shape:", df.shape)
print("Date range:", df["date"].min().date(), "to", df["date"].max().date())
print("Unique link IDs:", df["linkid"].nunique())

In [ ]:
# 3. Available traffic metrics (event-level)
event_counts = df["event"].value_counts()
event_share = (event_counts / len(df) * 100).round(2)
event_summary = pd.DataFrame({"Event count": event_counts, "Share (%)": event_share})
display(event_summary)

pageviews = int(event_counts.get("pageview", 0))
clicks = int(event_counts.get("click", 0))
previews = int(event_counts.get("preview", 0))
print(f"Click events / pageview events (descriptive ratio): {clicks/pageviews*100:.2f}%" if pageviews else "No pageviews")
print(f"Preview events / pageview events (descriptive ratio): {previews/pageviews*100:.2f}%" if pageviews else "No pageviews")

print("Sessions: unavailable — no session ID or inactivity-based sessionization fields.")
print("Unique users: unavailable — no user identifier.")
print("Bounce rate: unavailable — no session-level entry/interaction definition.")
print("Average session duration: unavailable — no event timestamp or duration.")
print("Landing/exit pages and referrers: unavailable — no page URL or referral-source field.")

In [ ]:
# 4. Daily event trends
daily = df.groupby(["date", "event"]).size().unstack(fill_value=0)
display(daily)
daily.plot(figsize=(10,4), marker="o", title="Daily Event Volume")
plt.ylabel("Events"); plt.xlabel("Date"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

In [ ]:
# 5. Geographic distribution
country_clicks = df[df["event"] == "click"].groupby("country").size().sort_values(ascending=False)
display(country_clicks.head(15).to_frame("Click events"))
country_clicks.head(10).sort_values().plot(kind="barh", figsize=(8,5), title="Top 10 Countries by Click Events")
plt.xlabel("Click events"); plt.tight_layout(); plt.show()

In [ ]:
# 6. Top content by click events
artist_clicks = df[df["event"] == "click"].groupby("artist").size().sort_values(ascending=False)
track_clicks = df[df["event"] == "click"].groupby("track").size().sort_values(ascending=False)
display(artist_clicks.head(10).to_frame("Click events"))
display(track_clicks.head(10).to_frame("Click events"))
artist_clicks.head(10).sort_values().plot(kind="barh", figsize=(8,5), title="Top 10 Artists by Click Events")
plt.xlabel("Click events"); plt.tight_layout(); plt.show()
track_clicks.head(10).sort_values().plot(kind="barh", figsize=(8,5), title="Top 10 Tracks by Click Events")
plt.xlabel("Click events"); plt.tight_layout(); plt.show()

In [ ]:
# 7. Link-level engagement (not unique-user conversion)
link_event = df.pivot_table(index="linkid", columns="event", values="date", aggfunc="count", fill_value=0)
for col in ["pageview", "click", "preview"]:
    if col not in link_event.columns:
        link_event[col] = 0
link_event["clicks_per_pageview_pct"] = np.where(
    link_event["pageview"] > 0, link_event["click"] / link_event["pageview"] * 100, np.nan
)
display(link_event.sort_values("click", ascending=False).head(15))
print("Caution: these are event-count ratios by link ID, not user-level conversion rates.")

# 8. Five optimization recommendations for Alfido Tech

1. **Instrument sessions and users:** Add pseudonymous user IDs, session IDs and precise event timestamps so sessions, unique users, bounce rate and duration can be measured correctly.
2. **Capture page and referral metadata:** Record page URL, landing page, exit page, referrer, campaign UTM parameters and channel to identify high-performing entry points and referral sources.
3. **Improve click-through from pageviews:** Compare pageview-to-click event volumes by link ID and test clearer calls to action, page layout and content placement. Treat the ratio as an event-level diagnostic, not a conversion rate.
4. **Personalize content by geography:** India, United States and Saudi Arabia lead click-event volume in this sample. Test localized content, language, timing and campaigns in these markets.
5. **Promote high-engagement content and monitor quality:** Feature top-clicked artists/tracks, but also track downstream outcomes (preview completion, sign-ups, purchases or other defined conversions) to avoid optimizing clicks alone.

## 9. Limitations
- No user ID, session ID, timestamp-of-day, page URL, referrer or duration.
- No direct way to identify user journeys, landing/exit pages, sessions, unique users, bounce rate or average session duration.
- Dataset covers only seven calendar days (19–25 August 2021), so it cannot establish long-term seasonality.
- `linkid` identifies a link, not a person or a session.